# Interval consistency losses: a medium-scale comparison

Trains the `mc` baseline against every objective in `consistency_losses.md` on the canonical maze and tracks
all of them against the exact test set during training.

| config | consistency term |
| --- | --- |
| `mc` | none -- the baseline |
| `mc_local` | mean of the one-step residuals, `delta_t^2` |
| `mc_all` | every interval, raw, via the `2(n+1)/n * Var(c)` shortcut |
| `mc_all_scaled` | the same divided by `s(n) = (n+2)/3` |
| `mc_mixed` | half local, half `all_scaled` |
| `mc_poly_len` | every interval weighted by its length (the O(dn) polynomial extension) |
| `mc_multiscale` | equal weight per power-of-two length, each divided by the length |

**Every config has an identical data loss.** The consistency term runs on its own batch of rollouts tokenized
in both modes, and contributes only its own gradient, so the only difference between a row and the baseline is
that one added term.

Each objective gets its own `lambda_cons` (`experiments.CONS_LAMBDA`) chosen so they start at roughly equal
gradient pull on the shared weights -- raw `all` and `poly_len` sit ~100x above the scaled family in loss
value, so equal lambdas would not be a fair comparison. Use `LAMBDA_SCALE` to move them all together.

1. **Runtime -> Change runtime type -> GPU.** Consistency runs cost about 2x the baseline (two passes per
   consistency rollout), so CPU is slow here.
2. Set `REPO` and `BRANCH`. For a **private** repo, add a Colab secret `GITHUB_TOKEN` with read access.
3. **Runtime -> Run all.** If an import error already happened this session, **Runtime -> Restart session** first.

Runs go to Google Drive (`RUNS_DIR`), so re-running after a disconnect skips finished runs and continues.

In [ ]:
REPO = "https://github.com/amdson/scrl.git"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
USE_DRIVE = True  #@param {type:"boolean"}
RUNS_DIR = "/content/drive/MyDrive/sillyrl/runs"  #@param {type:"string"}

In [ ]:
# Clone (or update) the repo and make sure the dependencies are importable.
import os, subprocess, sys

url = REPO
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
    if token:
        url = REPO.replace("https://", f"https://{token}@")
except Exception:
    pass  # no secret: public repo

if not os.path.exists("/content/sillyrl/.git"):
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, url, "/content/sillyrl"], check=True)
else:
    subprocess.run(["git", "-C", "/content/sillyrl", "pull", "-q"], check=True)
os.chdir("/content/sillyrl")
sys.path.insert(0, "/content/sillyrl")

# Colab's preinstalled flax can lag its JAX (e.g. flax calling jax.core APIs that JAX 0.11 removed), so always
# upgrade flax and optax before importing them. pip keeps Colab's JAX if it already satisfies them; if it had
# to upgrade JAX, bring the GPU plugin (jax-cuda*) to the same version so the runtime doesn't fall back to CPU.
import importlib.metadata as md
jax_before = md.version("jax")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "flax", "optax"], check=True)
jax_after = md.version("jax")
if jax_after != jax_before:
    plugins = sorted({d.metadata["Name"] for d in md.distributions()
                      if (d.metadata["Name"] or "").lower().replace("_", "-").startswith("jax-cuda")})
    if plugins:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *[f"{p}=={jax_after}" for p in plugins]], check=True)
    print(f"jax {jax_before} -> {jax_after}; plugins updated: {plugins}")

import jax, flax, optax
print("commit", subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip())
print("jax", jax.__version__, "| flax", flax.__version__, "| optax", optax.__version__, "|", jax.devices())

In [ ]:
# Where runs are saved. Must be set before importing maze_consistency.train.
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["RUNS_DIR"] = RUNS_DIR
else:
    os.environ["RUNS_DIR"] = "runs"
os.makedirs(os.environ["RUNS_DIR"], exist_ok=True)
print("runs ->", os.environ["RUNS_DIR"])

In [ ]:
# Rebuild the dataset (100k random walks) and the exact test set from data/canonical/maze.txt. Deterministic.
!python run.py dataset | tail -4
!python run.py testset | tail -1

## Run the comparison

`STEPS = 3000` at `BATCH = 32` is the medium-scale setting: on a Colab GPU the baseline is a couple of
minutes and each consistency run roughly twice that, so the full sweep is on the order of 20-30 minutes.
Drop `STEPS` to 500 for a quick shakedown first.

`CONS_BATCH` is how many rollouts the consistency term sees per step; each costs two forward passes.
`LAMBDA_SCALE` multiplies every per-objective `lambda_cons` at once -- set it to 0.1 or 10 to re-run the whole
comparison at a different consistency strength.

In [ ]:
STEPS = 3000  #@param {type:"integer"}
BATCH = 32  #@param {type:"integer"}
CONS_BATCH = 16  #@param {type:"integer"}
LAMBDA_SCALE = 1.0  #@param {type:"number"}
SEEDS = "0"  #@param {type:"string"}
EVAL_EVERY = 250  #@param {type:"integer"}
EVAL_PER_SETTING = 50  #@param {type:"integer"}
PREFIX = "cons"  #@param {type:"string"}

from dataclasses import replace
from maze_consistency import experiments as E

# Apply CONS_BATCH and LAMBDA_SCALE to every consistency config.
for name, cfg in E.CONS_CONFIGS.items():
    E.CONFIGS[name] = replace(cfg, w_cons=cfg.w_cons * LAMBDA_SCALE, cons_batch=CONS_BATCH)

sc = E.SweepConfig(configs=E.CONS_SWEEP, seeds=tuple(int(s) for s in SEEDS.split(",")),
                   steps=STEPS, batch=BATCH, eval_every=EVAL_EVERY,
                   eval_per_setting=EVAL_PER_SETTING, prefix=PREFIX)
print(sc)
for c in E.CONS_SWEEP:
    lc = E.CONFIGS[c]
    print(f"  {c:<16} cons={lc.cons!s:<5} loss={lc.cons_loss if lc.cons else '-':<12} lambda={lc.w_cons if lc.cons else 0:g}")
E.run_sweep(sc)

## Exact-test metrics

`act_kl` is KL(true action distribution || model) against the DP's exact answer, `value_kl` the same for the
value head. `bin 10` / `bin 11` are high-return conditions and `best far` is near-optimal behaviour from far
starts -- the setting the random-walk training data never demonstrates, so it is the one where a consistency
term has the most room to help. Lower is better everywhere.

In [ ]:
from IPython.display import Image, display
E.plot_sweep(sc.prefix)
display(Image(os.path.join(os.environ["RUNS_DIR"], sc.prefix, "curves.png")))

## Collapse diagnostics

Every one of these objectives has the same degenerate global optimum: `v_t == u_t` with `b_t` flat in `t` --
ignore R entirely -- makes every residual exactly zero. So a falling `cons` curve is not on its own good news.

Read the two right-hand panels first:

- **`cond_gap`** = mean `|v_t - u_t|`, how much conditioning on R changes the model's own trajectory
  likelihood. Decaying toward 0 means the R-conditioned and unconditioned policies have merged.
- **`info_gain`** = `log q_n(R) - log q_0(R)`, how much the reward head learns about R over a trajectory.
  Decaying toward 0 means the reward head has gone flat.

A run whose `cons` falls while either of those decays has bought agreement by discarding the reward channel:
its lambda is too high. Both diagnostics are logged, never optimized.

In [ ]:
E.plot_cons_diagnostics(sc.prefix)
display(Image(os.path.join(os.environ["RUNS_DIR"], sc.prefix, "cons_diagnostics.png")))

## Final numbers

Exact-test metrics at the last checkpoint, and the end-of-run diagnostics beside them. `delta` columns are
each config minus the `mc` baseline, so negative = better than baseline.

In [ ]:
import numpy as np

test, train_h = E.load_sweep(sc.prefix, "test"), E.load_sweep(sc.prefix, "train")
cols = [f"{m}/{s}" for m in ("act_kl", "value_kl") for s in ("NOR", "bin 10", "bin 11", "best far")]
diag = ("cons", "cond_gap", "info_gain")
mean_last = lambda hists, k: float(np.mean([h[-1].get(k, np.nan) for h in hists]))

rows = {c: ({k: mean_last(test[c], k) for k in cols} | {k: mean_last(train_h[c], k) for k in diag})
        for c in E.CONS_SWEEP if c in test}
base = rows["mc"]

w = max(len(c) for c in rows) + 2
print("exact-test KL in nats, lower is better. (d) = minus the mc baseline, so negative beats baseline.")
print("consistency term and its collapse diagnostics on the right; cond_gap / info_gain -> 0 means R ignored.")
for group in (cols[:4], cols[4:]):
    print()
    print(" " * w + "".join(f"{c.split('/')[1]:>22}" for c in group))
    print(f"{group[0].split('/')[0]:<{w}}" + "".join(f"{'value':>12}{'(d)':>10}" for _ in group))
    for c, r in rows.items():
        cells = "".join(f"{r[k]:12.4f}" + ("" if c == "mc" else f"{r[k] - base[k]:+10.4f}").rjust(10)
                        for k in group)
        print(f"{c:<{w}}" + cells)
print()
print(f"{'config':<{w}}" + "".join(f"{k:>14}" for k in diag))
for c, r in rows.items():
    print(f"{c:<{w}}" + "".join(f"{r[k]:14.4f}" if np.isfinite(r[k]) else f"{'-':>14}" for k in diag))

## Is the extra long-range weight earning anything?

`L_all` only beats `L_local` when residuals accumulate coherently rather than cancelling. The cheap test is
how the interval residual grows with interval length: `rms ~ sqrt(l)` means iid diffusion (nothing for the
long-range weighting to catch), `rms ~ l` means coherent drift (the regime `L_all` is designed for).

This measures it on the trained baseline, so you can see whether the ranking above should have been
predictable. On a short local run it comes out well below 0.5.

In [ ]:
import jax, numpy as np
from maze_consistency.dataset import load as load_data
from maze_consistency.tokens import Tokenizer
from maze_consistency.model import MazeTransformer
from maze_consistency.train import load_run, N_HELDOUT
import maze_consistency.consistency as C
import matplotlib.pyplot as plt

maze, d = load_data()
tok = Tokenizer(maze)
n_train = len(d["length"]) - N_HELDOUT
rng = np.random.default_rng(0)

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for c, cfg in enumerate(["mc", "mc_all_scaled"]):
    try:
        params, mcfg = load_run(f"{sc.prefix}/{cfg}_s{SEEDS.split(',')[0]}")
    except FileNotFoundError:
        continue
    terms_fn = C.make_terms_fn(MazeTransformer(mcfg), tok)
    b = C.rollout_batch(tok, maze, d, rng.choice(np.arange(n_train, n_train + N_HELDOUT), 64, replace=False))
    t, r, losses, diag = C.evaluate(params, terms_fn, b)
    ell, rms, _ = C.interval_stats_by_length(np.asarray(r["c"]), np.asarray(b["lengths"]))
    slope = np.polyfit(np.log(ell[1:]), np.log(rms[1:]), 1)[0]
    ax[0].loglog(ell, rms, lw=1.6, color=f"C{c}", label=f"{cfg}  (l^{slope:.2f})")
    ax[1].bar(np.arange(len(C.ALL)) + 0.4 * c - 0.2, [float(np.asarray(losses[k]).mean()) for k in C.ALL],
              0.4, color=f"C{c}", label=cfg)
    print(f"{cfg:<14} rms Delta ~ l^{slope:.2f}   cond_gap {float(diag['cond_gap'].mean()):.4f}"
          f"   info_gain {float(diag['info_gain'].mean()):.4f}")
ax[0].loglog(ell, rms[0] * np.sqrt(ell), "k:", lw=1, label="~sqrt(l)  iid diffusion")
ax[0].loglog(ell, rms[0] * ell, "k--", lw=1, label="~l  coherent drift")
ax[0].set_xlabel("interval length l"); ax[0].set_ylabel("rms Delta")
ax[0].set_title("interval residual by length, held out", fontsize=9)
ax[0].legend(fontsize=8); ax[0].grid(alpha=.3, which="both")
ax[1].set_xticks(range(len(C.ALL))); ax[1].set_xticklabels(list(C.ALL), rotation=40, ha="right", fontsize=8)
ax[1].set_yscale("log"); ax[1].set_title("every consistency loss, held out", fontsize=9)
ax[1].legend(fontsize=8); ax[1].grid(alpha=.3, axis="y")
plt.tight_layout(); plt.show()

## Hacking this

- **Sweep lambda instead of objective**: `E.CONFIGS.update(E.cons_lambda_configs("all_scaled", (0.1, 1, 10)))`
  then pass those names as `configs=` to `SweepConfig`. Or just change `LAMBDA_SCALE` and a fresh `PREFIX`.
- **More seeds**: `SEEDS = "0,1,2"`. The curve plots then show a min..max band rather than a single line.
- **A different data loss**: the consistency term is independent of `mc` -- `LossConfig(td=True, cons=True, ...)`
  pairs it with the TD value loss instead, and `a=True` adds identity (A) on top.
- **A new objective**: write a function of the `residuals()` dict, add it to `consistency.ALL` and give it a
  `CONS_LAMBDA` entry; it joins `CONS_SWEEP` automatically.
- **Locally**: `python run.py cons-sweep 3000` then `python run.py cons-plot`.